# AI Observability — Dev Log

## Objetivo e papel no pipeline

`core/ai_observability` instrumenta chamadas aos módulos do AthenaGov AI com
métricas estruturadas reais: latência (`time.perf_counter`, nunca mockado),
contagem e status (ok/erro) — exportáveis em formato de texto compatível com
o padrão de exposição do Prometheus.

**Escopo honesto**: não sobe um coletor/exporter de rede de verdade — o que
existe é o registro determinístico em memória e a serialização no formato de
texto que um scraper Prometheus entende. Ligar isso a um coletor real
(`opentelemetry-sdk`, endpoint `/metrics` em `core/governance_copilot`) é
TODO explícito de onda futura.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.ai_observability.observability import ObservabilityRecorder, export_prometheus_text, traced
from core.pii_detection.detector import detect
from core.prompt_security.scanner import scan

recorder = ObservabilityRecorder()
with traced(recorder, module="pii_detection", function="detect"):
    detect("CPF 111.444.777-35")
with traced(recorder, module="prompt_security", function="scan"):
    scan("Qual a previsão do tempo?")
try:
    with traced(recorder, module="pii_detection", function="detect"):
        raise RuntimeError("falha simulada de dependência externa")
except RuntimeError:
    pass

snapshot = recorder.snapshot()
print(f"Total de chamadas: {snapshot.total_calls} | erros: {snapshot.error_count} | duração média: {snapshot.avg_duration_ms:.4f}ms")
print("Por módulo:", snapshot.by_module)
print()
print("--- Exportação Prometheus ---")
print(export_prometheus_text(snapshot))

Total de chamadas: 3 | erros: 1 | duração média: 0.1041ms
Por módulo: {'pii_detection': 2, 'prompt_security': 1}

--- Exportação Prometheus ---
# HELP athenagov_module_calls_total Total de chamadas registradas por módulo.
# TYPE athenagov_module_calls_total counter
athenagov_module_calls_total{module="pii_detection"} 2
athenagov_module_calls_total{module="prompt_security"} 1
# HELP athenagov_module_call_errors_total Total de chamadas com erro.
# TYPE athenagov_module_call_errors_total counter
athenagov_module_call_errors_total 1
# HELP athenagov_module_call_duration_ms_avg Duração média (ms) das chamadas registradas.
# TYPE athenagov_module_call_duration_ms_avg gauge
athenagov_module_call_duration_ms_avg 0.1041



A terceira chamada propositalmente levanta uma exceção real dentro do bloco
`traced(...)` — o context manager registra `status="error"` com a mensagem
real da exceção e **relança** a exceção original (nunca a engole), então o
`try/except` do próprio código de demonstração precisa capturá-la. É esse
comportamento que os testes (`test_traced_records_error_and_reraises`)
verificam.

## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/ai_observability/tests -v
```

7 testes: chamada real a `pii_detection.detect()` registrada com sucesso;
exceção real registrada como erro e relançada; média de duração e contagem
por módulo com `time.sleep()` real; snapshot vazio; reset; formato de
exportação Prometheus (inclusive snapshot vazio).

## Handoff Summary

- **Status:** ✅ done — 7/7 testes passando.
- **Consumível por:** `core/governance_copilot`, que poderia envolver cada
  endpoint com `traced(...)` e expor `export_prometheus_text` num futuro
  `GET /metrics` — não implementado nesta onda (TODO).